In [2]:
import os 
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [3]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

In [4]:
!pip install langchain-groq

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi ,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [6]:
video_id = "bYWOWwVQtBo" # only the ID, not full URL

try:
  ytt_api = YouTubeTranscriptApi()  # create an instance first
  transcript_list = ytt_api.fetch(video_id , languages=['hi']) 


  #Flatten to plain text 
  transcript = " ".join(chunk.text for chunk in transcript_list)
  print(transcript)

except TranscriptsDisabled :
  print("No english Captions are available for this video")
  

एक नॉर्मल दिन में वीजी सिद्धार्थ के कॉनबॉय में दो गाड़ियां होती थी। एक उनकी खुद की गाड़ी और दूसरी बैकअप के लिए। पर 29 जुलाई को सिद्धार्थ जब बेंगलुरु से अपने घर से निकलते हैं तो दो चीजें डिफरेंट है। पहली बस वो अपनी Toyota Innov गाड़ी के साथ आए हैं और दूसरी उनके रेगुलर ड्राइवर रवि लीफ पर है। इसीलिए एक नया ड्राइवर ये गाड़ी चला रहा है। घर निकलने से पहले सिद्धार्थ अपने परिवार को बुलाते हैं कि वो कुछ कॉफी एस्टेट्स विजिट करने जा रहे हैं। या तो सकलेशपुरा वाला या फिर मुदगेरे वाला। दोनों जो चिकमगलुरू इलाके में हैं। पर सकलेशपुरा पहुंचने से पहले ही वह ड्राइवर को बोलते हैं बेंगलुरु की तरफ बढ़ने के लिए। फिर नेत्रावती नदी पर एक ब्रिज पर वो रोकने को बोलते हैं। गाड़ी से निकलते हैं और ड्राइवर को बोलते हैं कि तुम ब्रिज पार कर लो। मैं उधर चलता रहूंगा। फिर वो ऑपोजिट डायरेक्शन में चलने लग जाते हैं जब तक ड्राइवर उनको देख भी नहीं सकता। ड्राइवर कई घंटे वेट करता है। पर जब 8:00 बज जाते हैं और सूरज भी डूब जाता है वो सिद्धार्थ को कॉल करने की कोशिश करते हैं। पर कोई जवाब भी नहीं देता। इसीलिए नेक्स्ट दिन एक मिसिंग कं

In [7]:
print("1 -->" , end = " ")

for i in range(1,5):
  print(transcript_list[i].text , end = f" \n{i+1} --> ")

1 --> कॉनबॉय में दो गाड़ियां होती थी। एक उनकी 
2 --> खुद की गाड़ी और दूसरी बैकअप के लिए। पर 
3 --> 29 जुलाई को सिद्धार्थ जब बेंगलुरु से 
4 --> अपने घर से निकलते हैं तो दो चीजें 
5 --> 

# Indexing (Text Splitting))

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunks = splitter.create_documents([transcript])
chunks[0].page_content

'एक नॉर्मल दिन में वीजी सिद्धार्थ के कॉनबॉय में दो गाड़ियां होती थी। एक उनकी खुद की गाड़ी और दूसरी बैकअप के लिए। पर 29 जुलाई को सिद्धार्थ जब बेंगलुरु से अपने घर से निकलते हैं तो दो चीजें डिफरेंट है। पहली बस वो अपनी Toyota Innov गाड़ी के साथ आए हैं और दूसरी उनके रेगुलर ड्राइवर रवि लीफ पर है। इसीलिए एक नया ड्राइवर ये गाड़ी चला रहा है। घर निकलने से पहले सिद्धार्थ अपने परिवार को बुलाते हैं कि वो कुछ कॉफी एस्टेट्स विजिट करने जा रहे हैं। या तो सकलेशपुरा वाला या फिर मुदगेरे वाला। दोनों जो चिकमगलुरू इलाके में हैं। पर सकलेशपुरा पहुंचने से पहले ही वह ड्राइवर को बोलते हैं बेंगलुरु की तरफ बढ़ने के लिए। फिर नेत्रावती नदी पर एक ब्रिज पर वो रोकने को बोलते हैं। गाड़ी से निकलते हैं और ड्राइवर को बोलते हैं कि तुम ब्रिज पार कर लो। मैं उधर चलता रहूंगा। फिर वो ऑपोजिट डायरेक्शन में चलने लग जाते हैं जब तक ड्राइवर उनको देख भी नहीं सकता। ड्राइवर कई घंटे वेट करता है। पर जब 8:00 बज जाते हैं और सूरज भी डूब जाता है वो सिद्धार्थ को कॉल करने की कोशिश करते हैं। पर कोई जवाब भी नहीं देता। इसीलिए नेक्स्ट दिन एक मिसिंग'

In [9]:
len(chunks)

26

In [10]:
chunks[0]

Document(metadata={}, page_content='एक नॉर्मल दिन में वीजी सिद्धार्थ के कॉनबॉय में दो गाड़ियां होती थी। एक उनकी खुद की गाड़ी और दूसरी बैकअप के लिए। पर 29 जुलाई को सिद्धार्थ जब बेंगलुरु से अपने घर से निकलते हैं तो दो चीजें डिफरेंट है। पहली बस वो अपनी Toyota Innov गाड़ी के साथ आए हैं और दूसरी उनके रेगुलर ड्राइवर रवि लीफ पर है। इसीलिए एक नया ड्राइवर ये गाड़ी चला रहा है। घर निकलने से पहले सिद्धार्थ अपने परिवार को बुलाते हैं कि वो कुछ कॉफी एस्टेट्स विजिट करने जा रहे हैं। या तो सकलेशपुरा वाला या फिर मुदगेरे वाला। दोनों जो चिकमगलुरू इलाके में हैं। पर सकलेशपुरा पहुंचने से पहले ही वह ड्राइवर को बोलते हैं बेंगलुरु की तरफ बढ़ने के लिए। फिर नेत्रावती नदी पर एक ब्रिज पर वो रोकने को बोलते हैं। गाड़ी से निकलते हैं और ड्राइवर को बोलते हैं कि तुम ब्रिज पार कर लो। मैं उधर चलता रहूंगा। फिर वो ऑपोजिट डायरेक्शन में चलने लग जाते हैं जब तक ड्राइवर उनको देख भी नहीं सकता। ड्राइवर कई घंटे वेट करता है। पर जब 8:00 बज जाते हैं और सूरज भी डूब जाता है वो सिद्धार्थ को कॉल करने की कोशिश करते हैं। पर कोई जवाब भी नहीं द

# Embedding Generation and Vector Stores 

In [12]:
!pip install sentence-transformers

In [13]:
embeddings = HuggingFaceBgeEmbeddings()
vector_stores = FAISS.from_documents(chunks , embeddings)

C:\Users\harsh\AppData\Local\Temp\ipykernel_29812\3904160196.py:1: LangChainDeprecationWarning: Default values for HuggingFaceBgeEmbeddings.model_name were deprecated in LangChain 0.2.5 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceBgeEmbeddings constructor instead.
  embeddings = HuggingFaceBgeEmbeddings()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
(vector_stores.index_to_docstore_id)

{0: 'd4679653-335a-4d1e-98ee-981b8c47b4c7',
 1: '409de785-535e-4bf2-99d8-4c9dc3b939f6',
 2: '0ae0512a-c771-4807-afed-c59c82c3a010',
 3: 'e38295d9-6ab4-4b05-b920-f35dff3453d5',
 4: '870c1105-a802-43c3-8f34-2545f8b7ca00',
 5: '099ccf32-000a-423d-9930-f5cb9aa381f7',
 6: '389b4ead-b214-42c4-a8b8-68529eae641b',
 7: '20961e2a-5c9b-47f8-87b3-c197929204ac',
 8: '8e3c9b10-afc5-4d29-9a30-0e108d652e50',
 9: 'f48c9b63-0638-49a0-9a30-3804e0f19e90',
 10: '37e3048f-8570-493e-9791-31fac065ce4f',
 11: 'af0eff0d-6f4f-4855-b66c-5b78d1f76db1',
 12: '365dd10f-f209-44a6-aeaa-dfa394c18b1a',
 13: '637eb9c6-3315-493e-be5f-44aebe7b67f5',
 14: 'f12b5ccc-971e-4a83-8905-07ed69be3a75',
 15: 'bacdd8e5-716b-4d30-8013-20cfd8e6c654',
 16: '953f1d0e-1012-4e7f-992c-87df89609779',
 17: '5c1381d3-51dd-4daa-9534-bf5c94451e69',
 18: '6ebf241f-43b1-4699-8aef-b15f4146ab87',
 19: '9a8f1ff9-f4e1-4302-8a0f-5e3b55fdc077',
 20: '614e483e-a9aa-4a3b-9071-23b2e13df712',
 21: '3721f2bf-cd03-4ed7-a765-7c8d4836b7ae',
 22: '660204f3-12ca-

In [15]:
vector_stores.get_by_ids(['359db823-e110-4c91-856d-8b9525ba7960'])

[]

# Retriever 

In [16]:
retriever = vector_stores.as_retriever(search_type = "similarity" , search_kwargs = {"k":4})

In [17]:
retriever.invoke("What is he talking about in the video ?")

[Document(id='d4679653-335a-4d1e-98ee-981b8c47b4c7', metadata={}, page_content='एक नॉर्मल दिन में वीजी सिद्धार्थ के कॉनबॉय में दो गाड़ियां होती थी। एक उनकी खुद की गाड़ी और दूसरी बैकअप के लिए। पर 29 जुलाई को सिद्धार्थ जब बेंगलुरु से अपने घर से निकलते हैं तो दो चीजें डिफरेंट है। पहली बस वो अपनी Toyota Innov गाड़ी के साथ आए हैं और दूसरी उनके रेगुलर ड्राइवर रवि लीफ पर है। इसीलिए एक नया ड्राइवर ये गाड़ी चला रहा है। घर निकलने से पहले सिद्धार्थ अपने परिवार को बुलाते हैं कि वो कुछ कॉफी एस्टेट्स विजिट करने जा रहे हैं। या तो सकलेशपुरा वाला या फिर मुदगेरे वाला। दोनों जो चिकमगलुरू इलाके में हैं। पर सकलेशपुरा पहुंचने से पहले ही वह ड्राइवर को बोलते हैं बेंगलुरु की तरफ बढ़ने के लिए। फिर नेत्रावती नदी पर एक ब्रिज पर वो रोकने को बोलते हैं। गाड़ी से निकलते हैं और ड्राइवर को बोलते हैं कि तुम ब्रिज पार कर लो। मैं उधर चलता रहूंगा। फिर वो ऑपोजिट डायरेक्शन में चलने लग जाते हैं जब तक ड्राइवर उनको देख भी नहीं सकता। ड्राइवर कई घंटे वेट करता है। पर जब 8:00 बज जाते हैं और सूरज भी डूब जाता है वो सिद्धार्थ को कॉल क

# Augmentaion

In [18]:
from dotenv import load_dotenv
import os
load_dotenv()

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",  # specify model
    api_key= os.getenv("GROQ_API_KEY"),  # your groq key
    temperature=0.7
)

In [19]:
prompt = PromptTemplate(
    template = """ You are a helpful assistant who respond in a very fowmal way.
    You shold answer the question based on the following retrieved information from a youtube video transcript.
    If you don't know the answer, say you don't know.

    {context}
    Question : {question}
    """,
    input_variables = ['context', 'question']
)

In [20]:
question = "What are they talking about in the video ?"
retrieved_docs = retriever.invoke(question)
retrieved_docs


[Document(id='d4679653-335a-4d1e-98ee-981b8c47b4c7', metadata={}, page_content='एक नॉर्मल दिन में वीजी सिद्धार्थ के कॉनबॉय में दो गाड़ियां होती थी। एक उनकी खुद की गाड़ी और दूसरी बैकअप के लिए। पर 29 जुलाई को सिद्धार्थ जब बेंगलुरु से अपने घर से निकलते हैं तो दो चीजें डिफरेंट है। पहली बस वो अपनी Toyota Innov गाड़ी के साथ आए हैं और दूसरी उनके रेगुलर ड्राइवर रवि लीफ पर है। इसीलिए एक नया ड्राइवर ये गाड़ी चला रहा है। घर निकलने से पहले सिद्धार्थ अपने परिवार को बुलाते हैं कि वो कुछ कॉफी एस्टेट्स विजिट करने जा रहे हैं। या तो सकलेशपुरा वाला या फिर मुदगेरे वाला। दोनों जो चिकमगलुरू इलाके में हैं। पर सकलेशपुरा पहुंचने से पहले ही वह ड्राइवर को बोलते हैं बेंगलुरु की तरफ बढ़ने के लिए। फिर नेत्रावती नदी पर एक ब्रिज पर वो रोकने को बोलते हैं। गाड़ी से निकलते हैं और ड्राइवर को बोलते हैं कि तुम ब्रिज पार कर लो। मैं उधर चलता रहूंगा। फिर वो ऑपोजिट डायरेक्शन में चलने लग जाते हैं जब तक ड्राइवर उनको देख भी नहीं सकता। ड्राइवर कई घंटे वेट करता है। पर जब 8:00 बज जाते हैं और सूरज भी डूब जाता है वो सिद्धार्थ को कॉल क

In [21]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'एक नॉर्मल दिन में वीजी सिद्धार्थ के कॉनबॉय में दो गाड़ियां होती थी। एक उनकी खुद की गाड़ी और दूसरी बैकअप के लिए। पर 29 जुलाई को सिद्धार्थ जब बेंगलुरु से अपने घर से निकलते हैं तो दो चीजें डिफरेंट है। पहली बस वो अपनी Toyota Innov गाड़ी के साथ आए हैं और दूसरी उनके रेगुलर ड्राइवर रवि लीफ पर है। इसीलिए एक नया ड्राइवर ये गाड़ी चला रहा है। घर निकलने से पहले सिद्धार्थ अपने परिवार को बुलाते हैं कि वो कुछ कॉफी एस्टेट्स विजिट करने जा रहे हैं। या तो सकलेशपुरा वाला या फिर मुदगेरे वाला। दोनों जो चिकमगलुरू इलाके में हैं। पर सकलेशपुरा पहुंचने से पहले ही वह ड्राइवर को बोलते हैं बेंगलुरु की तरफ बढ़ने के लिए। फिर नेत्रावती नदी पर एक ब्रिज पर वो रोकने को बोलते हैं। गाड़ी से निकलते हैं और ड्राइवर को बोलते हैं कि तुम ब्रिज पार कर लो। मैं उधर चलता रहूंगा। फिर वो ऑपोजिट डायरेक्शन में चलने लग जाते हैं जब तक ड्राइवर उनको देख भी नहीं सकता। ड्राइवर कई घंटे वेट करता है। पर जब 8:00 बज जाते हैं और सूरज भी डूब जाता है वो सिद्धार्थ को कॉल करने की कोशिश करते हैं। पर कोई जवाब भी नहीं देता। इसीलिए नेक्स्ट दिन एक मिसिंग\n

In [22]:
final_prompt=prompt.invoke({"context" : context_text , "question" : question})

# Generation

In [23]:
answer = llm.invoke(final_prompt)
print(answer.content)

वे सिद्धार्थ शिरोडकर और उनके कॉफी डे के बिजनेस के बारे में बात कर रहे हैं।


# Building a Chain

In [24]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [25]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text
    

In [26]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})


In [27]:
parallel_chain.invoke("What is Demis")

{'context': 'उस जमाने में लोगों के घरों में इंटरनेट नहीं था। सो इसीलिए 11 जुलाई 1986 में कैफे कॉफी डे का पहला आउटलेट खुला सेंट्रल बेंगलुरु के ब्रिगेड रोड पर और इस कैफे में इंटरनेट अवेलेबल था। सीसीडी में आपको इटालियन स्टाइल ड्रिंक्स जैसे कैपचीनोस एस्प्रेसोर्स और लाटे मिलती है। कई चीजें जिसको इंडियंस ने कभी एक्सपीरियंस नहीं किया था। ये कैफे एक इंस्टेंट हिट बन गया। अगले कुछ सालों के दौरान सिद्धार्थ ने फिर कई आउटलेट्स खोले और 2001 तक तो देश में 18 कैफेस खुल चुके थे मेनली बेंगलुरु, चेन्नई और मैसूर में। पर सिद्धार्थ अभी भी इस कंपनी को एक फैमिली बिजनेस की तरह चला रहे थे। मोस्टली एम्प्लाइज़ उनके फैमिली एसोसिएट्स थे और दूसरे कुछ दोस्त थे। जबभी भी कोई यंगस्टर उनके होमटाउन से नौकरी की बात करता वो उनको नौकरी देते थे। एंड दिस इज द स्टेज व्हेन नरेश मल्होत्रा एंटर्स द पिक्चर। नरेश ने पहले विजय माला के साथ काम किया था Berger एंड Paints जैसी कंपनीज़ को सेटअप करने के लिए और फिर बाद में KPMG के ऑपरेशन सेटअप करने के लिए। 2001 में नरेश CCD के सीईओ बन गए। कंपनी में 70 से 80 लोग थे बट द नंबर्स रियली शॉक्ड हिम।

In [28]:
parser = StrOutputParser()

In [29]:
main_chain = parallel_chain | prompt | llm | parser 

In [30]:
print(main_chain.invoke("summarise my video"))

प्रतिवेदन: 

वीडियो एक सिद्धार्थ की कहानी को दिखाता है, जिन्होंने अपने परिवार के कॉफी बिज़नेस को एक विशाल कंपनी में बदल दिया। सिद्धार्थ ने 2016 में आईआईटी कानपुर में एक इंडियन एंटरप्रेन्योरशिप समिट में भाग लिया, जहां उन्होंने कहा कि वे कार्ल मार्क्स की प्रिंसिपल्स से प्रभावित थे।

सिद्धार्थ ने कॉफी के बिज़नेस में बहुत पॉलिटिक्स और सरकारी इन्वॉल्वमेंट से बचने के लिए स्टॉक ब्रोकिंग में काम करना शुरू किया और अंत में अपनी कंपनी सिवान सिक्योरिटीज बनाई।

वे एक एक्सपोर्ट कंपनी एबीसीटीएल के माध्यम से कॉफी का व्यापार करते थे, जिससे उन्हें ₹55 प्रति किलोग्राम का मूल्य मिलता था। एबीसीटीएल की सफलता के बाद, सिद्धार्थ ने दूसरे ग्रोअर्स को ट्रेनिंग देना शुरू किया और उन्हें कैसे अपनी क्वालिटी बढ़ानी है और इंटरनेशनल मार्केट में क्या कीमतें हैं।

हालांकि, उनकी सफलता के बाद, सिद्धार्थ की मौत 29 जुलाई 2016 को नेत्रावती नदी में पाई गई, जो एक बड़ा मामला बन गया। उनके नोट में पता चला कि उन्होंने अपने एसेट्स को बेचने का फैसला किया था और अपने डेप्ट को कम करने का प्रयास किया था।

सिद्धार्थ की विधवा मालविका ने उनकी